Setup & Imports

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
from src.similarity.vectorizer import UnifiedVectorizer
from src.similarity.similarity_scorer import UnifiedScorer
from src.similarity.batch_comparator import compare_students_to_master
from src.utils.config_loader import load_config

config = load_config()
print(f"Active Similarity Method: {config['similarity']['method'].upper()}")
print(f"Active Model: {config['similarity'].get('semantic_model')}")

Test Unified Vectorizer on GPU / CPU

In [ ]:
vectorizer = UnifiedVectorizer()
scorer = UnifiedScorer()

docs = [
    "The cell membrane regulates substances entering and leaving the cell.", # Master
    "The plasma membrane controls what passes in and out of the cell.",     # Paraphrase (Good/Excellent)
    "Quantum entanglement describes correlated states of distant particles." # Unrelated (Poor)
]

vectors = vectorizer.transform(docs)
print(f"Vector representation type: {type(vectors)}")

# Compute similarity against master
sim_paraphrase = scorer.compute_score(vectors[0:1], vectors[1:2])
sim_unrelated = scorer.compute_score(vectors[0:1], vectors[2:3])

print(f"Paraphrase Score: {sim_paraphrase:.4f} -> {scorer.match_level(sim_paraphrase)}")
print(f"Unrelated Score:  {sim_unrelated:.4f} -> {scorer.match_level(sim_unrelated)}")

Full Batch Pipeline with Sample Data

In [ ]:
master_file = "../data/samples/ANS_PDF.pdf"
student_files = [
    "../data/samples/ANS_DOCX.docx",
    "../data/samples/ANS_TXT.txt",
    "../data/samples/ANS_PNG.png"
]

df_results = compare_students_to_master(student_files, master_file)
df_results

Test Local LLM Reasoning / Grading Feedback

In [ ]:
if config.get("llm_reasoning", {}).get("enabled", False):
    print("Testing Local LLM Feedback (Ollama)...")
    from src.extraction.extractor import extract_text
    
    master_txt = extract_text(master_file)["text"]
    student_txt = extract_text(student_files[1])["text"] # TXT variant
    score = df_results.loc[df_results['filename'] == student_files[1], 'similarity_score'].values[0]
    
    feedback = scorer.generate_feedback(master_txt, student_txt, score)
    print(f"\n--- AI Examiner Feedback ---\n{feedback}")
else:
    print("LLM reasoning disabled in config.yaml")